# Context OverFlow Problem Solution1

In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_groq import ChatGroq
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages.utils import trim_messages, count_tokens_approximately 
from dotenv import load_dotenv
import sqlite3
import os

# Example Testing count_tokens_approximately()

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.messages.utils import count_tokens_approximately

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the distance to the moon?"),
]

# Calculate approximate token count
estimated_tokens = count_tokens_approximately(messages)
print(f"Estimated tokens: {estimated_tokens}")

messages = trim_messages(
    messages=messages,
    max_tokens=20,
    token_counter=count_tokens_approximately,
    strategy='last'
)

print(messages)

print("Toal Messages Token  After Trimming: ",count_tokens_approximately(messages))

Estimated tokens: 25
[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={})]
Toal Messages Token  After Trimming:  12


In [2]:
load_dotenv()

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = os.getenv('GROQ_API_KEY')
)

In [3]:
MAX_TOKENS = 150

In [9]:
def chat_node(state: MessagesState):

    messages = state['messages']
    print("---------- Before Trimming ----------------")
    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)
    print()
    messages = trim_messages(
        state['messages'],
        strategy='last',
        token_counter=count_tokens_approximately,
        max_tokens= MAX_TOKENS

    )

    print("---------- After Trimming ----------------")
    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    result = llm.invoke(messages)

    return {
        'messages': [result]
    }

In [10]:
conn = sqlite3.connect(database='stm_db.db', check_same_thread=False)

check_pointer = SqliteSaver(conn=conn)

builder = StateGraph(MessagesState)


builder.add_node('chat_node', chat_node)

builder.add_edge(START, 'chat_node')
builder.add_edge('chat_node', END)

graph = builder.compile(checkpointer=check_pointer)

In [11]:
CONFIG = {
    'configurable': {
        'thread_id': 'thread_1'
    }
}

In [12]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': 'What is My Name'
            }
        ]
    },
    config=CONFIG
)

print(result['messages'][-1].content)

---------- Before Trimming ----------------
Current Token Count -> 16
What is My Name
What is My Name

---------- After Trimming ----------------
Current Token Count -> 16
What is My Name
What is My Name
I don't have any information about your name. I'm a large language model, I don't have the ability to know or recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!


In [13]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "I'm Imran Butt?"
            }
        ]
    },
    config=CONFIG
)

print(result['messages'][-1].content)

---------- Before Trimming ----------------
Current Token Count -> 130
What is My Name
What is My Name
I don't have any information about your name. I'm a large language model, I don't have the ability to know or recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!
I'm Imran Butt?

---------- After Trimming ----------------
Current Token Count -> 130
What is My Name
What is My Name
I don't have any information about your name. I'm a large language model, I don't have the ability to know or recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you an

In [14]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "Tell Me Who I Am?"
            }
        ]
    },
    config=CONFIG
)

print(result['messages'][-1].content)

---------- Before Trimming ----------------
Current Token Count -> 212
What is My Name
What is My Name
I don't have any information about your name. I'm a large language model, I don't have the ability to know or recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!
I'm Imran Butt?
Nice to meet you, Imran Butt! It's great that you've shared your name with me. I'll do my best to address you by your name in our conversation. How can I assist you today, Imran? Do you have any questions, topics you'd like to discuss, or just want to chat? I'm all ears!
Tell Me Who I Am?

---------- After Trimming ----------------
Current Token Count -> 90
I'm Imran Butt?
Nice to meet you, Imran Butt! It's great that you've shared your name with me. I'll do my best to add

In [15]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "Call My Name?"
            }
        ]
    },
    config=CONFIG
)

print(result['messages'][-1].content)

---------- Before Trimming ----------------
Current Token Count -> 483
What is My Name
What is My Name
I don't have any information about your name. I'm a large language model, I don't have the ability to know or recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!
I'm Imran Butt?
Nice to meet you, Imran Butt! It's great that you've shared your name with me. I'll do my best to address you by your name in our conversation. How can I assist you today, Imran? Do you have any questions, topics you'd like to discuss, or just want to chat? I'm all ears!
Tell Me Who I Am?
A philosophical question, Imran! Based on our conversation, I can tell you that you are:

1. **A person with a name**: You've shared your name with me, which is Imran Butt.
2. **Someone w

In [16]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "What is My Name?"
            }
        ]
    },
    config=CONFIG
)

print(result['messages'][-1].content)

---------- Before Trimming ----------------
Current Token Count -> 718
What is My Name
What is My Name
I don't have any information about your name. I'm a large language model, I don't have the ability to know or recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation!
I'm Imran Butt?
Nice to meet you, Imran Butt! It's great that you've shared your name with me. I'll do my best to address you by your name in our conversation. How can I assist you today, Imran? Do you have any questions, topics you'd like to discuss, or just want to chat? I'm all ears!
Tell Me Who I Am?
A philosophical question, Imran! Based on our conversation, I can tell you that you are:

1. **A person with a name**: You've shared your name with me, which is Imran Butt.
2. **Someone w

# Now At that moment we lost the context (My Name) and that is the problem with Trimming